In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from scipy.special import logit, expit
import pickle
from pathlib import Path

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[0] / '2_Propensities'))
sys.path.insert(0, str(Path.cwd().resolve().parents[0] / '4_Baselines' / '4.1_Matrix_Factorization'))

import SASRec_class as sasrec
import MF_class as MF

# 1 Loading Dataset and Propensities Model

In [ ]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Sequels'

# Load Oracle
oracle = pd.read_csv(data_path / 'oracle.csv')

# Load Chosen Pairs
chosen_pairs_dir = base_artifacts / 'Chosen_Pairs' / 'sequels'
with open(chosen_pairs_dir / 'chosen_pairs_ids.pkl', 'rb') as f:
    chosen_pairs_ids = pickle.load(f)

oracle[["cause_id", "effect_id"]] = np.asarray(chosen_pairs_ids)


# Load Data
data = pd.read_csv(base_artifacts / 'Datasets' / 'Processed' / 'goodreads' / 'data_clean.csv')

n_users = len(data['user_id'].unique())
n_items = len(data['item_id'].unique())

print(f"Number of users: {n_users:,}")
print(f"Number of items: {n_items:,}")

Number of users: 7,801
Number of items: 6,384


In [3]:
MF_model = MF.MatrixFactorizationTorch(n_users, n_items, n_factors=50)
MF_model.load(path=base_artifacts / 'MF_Models' / 'MF_model_goodreads.pt')
MF_model.eval()

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            7801
Number of items:            6384
Number of factors:          50
Learning rate:              0.001
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           20
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-07-18 10:52:15


MatrixFactorizationTorch()

# 2 Defining Baselines

In [4]:
data['interaction'] = 1
pivot_real = data.pivot(index='user_id', columns='item_id', values='interaction').fillna(0)
itemid_to_colidx_pivot_real = {item_id: col_idx for col_idx, item_id in enumerate(pivot_real.columns)}
pivot_real_np = pivot_real.values

Q_normalized = (MF_model.Q / torch.norm(MF_model.Q, dim=1, keepdim=True)).cpu().detach().numpy()

def cosimilarity(idx1, idx2):
    """Calculate cosine similarity between two items."""
    return np.dot(Q_normalized[idx1], Q_normalized[idx2])

def correlation(idx1, idx2):
    """Calculate correlation between two items."""
    colidx1 = itemid_to_colidx_pivot_real[idx1]
    colidx2 = itemid_to_colidx_pivot_real[idx2]
    T = pivot_real_np[:, colidx1]
    Y = pivot_real_np[:, colidx2]
    if T.std() == 0 or Y.std() == 0:
        return 0
    return np.corrcoef(T, Y)[0, 1]

def diff_of_conditionals(idx1, idx2):
    """Calculate difference of conditionals P(Y|T) - P(Y|~T) between two items."""
    colidx1 = itemid_to_colidx_pivot_real[idx1]
    colidx2 = itemid_to_colidx_pivot_real[idx2]
    T = pivot_real_np[:, colidx1]
    Y = pivot_real_np[:, colidx2]
    p_T = np.clip(np.mean(T), 1e-6, 1-1e-6)
    p_Y = np.mean(Y)
    p_TY = np.mean(T * Y)
    return p_TY / p_T - (p_Y - p_TY) / (1 - p_T)

def jacard_index(idx1, idx2):
    """Calculate Jaccard index between two items."""
    colidx1 = itemid_to_colidx_pivot_real[idx1]
    colidx2 = itemid_to_colidx_pivot_real[idx2]
    T = pivot_real_np[:, colidx1]
    Y = pivot_real_np[:, colidx2]
    intersection = np.sum((T > 0) & (Y > 0))
    union = np.sum((T > 0) | (Y > 0))
    if union == 0:
        return 0
    return intersection / union

# 3 Defining ATE

In [5]:
window_path = data_path / 'evaluation_data.pkl'
with open(window_path, 'rb') as f:
    ATE_window_data = pickle.load(f)

A_to_unique_A_idx = ATE_window_data['A_to_unique_A_idx']
treatment_items = ATE_window_data['treatment_items']
future_items  = ATE_window_data['future_items']
calibrated_logits_matrix = ATE_window_data['calibrated_logits_matrix']

In [6]:
def get_ATE(T, Y, pi, stabilized=False,):
    """
    Estimate ATE using IPW

    Parameters
    ----------
    - T : Treatment assignment array.
    - Y : Outcome array.
    - pi : Propensity score array.
    - stabilized : Whether to use stabilized weights in the IPW estimation. Default is False.
    """

    # --------------------------------------------------
    # IPW components 
    # --------------------------------------------------

    D1 = T / pi
    D0 = (1 - T) / (1 - pi)
    N1 = Y * D1
    N0 = Y * D0

    mN1 = N1.mean()
    mN0 = N0.mean()
    mD1 = D1.mean()
    mD0 = D0.mean()

    ESS_1 = (D1.sum() ** 2) / (np.sum(D1 ** 2) + 1e-12)
    ESS_0 = (D0.sum() ** 2) / (np.sum(D0 ** 2) + 1e-12)
    ESS_ATE = 2.0 / (1.0 / (ESS_1 + 1e-12) + 1.0 / (ESS_0 + 1e-12))

    # --------------------------------------------------
    # Point estimate
    # --------------------------------------------------

    eps = 1e-12
    if stabilized:
        term_1 = mN1 / (mD1 + eps)
        term_0 = mN0 / (mD0 + eps)
    else:
        term_1 = mN1
        term_0 = mN0
    
    ATE =  term_1 - term_0

    # --------------------------------------------------
    # Variance (delta method)
    # --------------------------------------------------

    n = len(T)
    if stabilized:
        Z = np.column_stack([N1, D1, N0, D0])
        g = np.array([
            1.0 / (mD1 + eps),
            -mN1 / ((mD1 + eps)**2),
            -1.0 / (mD0 + eps),
            mN0 / ((mD0 + eps)**2),
        ])
        S = np.cov(Z, rowvar=False, ddof=1)
        var_hat = (g @ S @ g) / n

    else:
        Z = np.column_stack([N1, N0])
        g = np.array([1.0, -1.0])
        S = np.cov(Z, rowvar=False, ddof=1)
        var_hat = (g @ S @ g) / n

    STD = float(np.sqrt(max(var_hat, 0.0)))

    return {
        "ATE": ATE, 
        "STD": STD,
        "size": n,
        "ESS_0": ESS_0,
        "ESS_1": ESS_1,
        "ESS_ATE": ESS_ATE,
    }

In [7]:
def get_ATE_wrapper(
    cause_item,
    effect_item,
    clip=0,
):
    """
    Estimate ATE using IPW

    Parameters
    ----------
    - cause_item : The item ID of the cause item.
    - effect_item : The item ID of the effect item.
    - clip : The clipping value for propensity scores to avoid extreme weights. Default is 0 (no clipping).
    - stabilized : Whether to use stabilized weights in the IPW estimation. Default is False.    
    """

    # -------------------------------------------------------------------
    # Calculate treatment and outcome variables, and propensity scores
    # -------------------------------------------------------------------

    T = treatment_items == cause_item
    Y = np.any(future_items == effect_item, axis=1)
    
    pi = expit(
        calibrated_logits_matrix[:, A_to_unique_A_idx[cause_item]]
    )
    pi = np.clip(pi, clip, 1 - clip)

    # --------------------------------------------------
    # Estimate ATE using IPW
    # --------------------------------------------------

    Horvitz_Thompson = get_ATE(T, Y, pi, stabilized=False)

    Hájek = get_ATE(T, Y, pi, stabilized=True)

    ablt_pi = 0.5 * np.ones_like(pi)
    ablation = get_ATE(T, Y, ablt_pi, stabilized=True)

    return {
        "ATE": Horvitz_Thompson["ATE"],
        "STD": Horvitz_Thompson["STD"],
        "ATE_STABILIZED": Hájek["ATE"],
        "STD_STABILIZED": Hájek["STD"],
        "ATE_ABLT": ablation["ATE"],
        "STD_ABLT": ablation["STD"],
        "size": Horvitz_Thompson["size"],
        "ESS_0": Horvitz_Thompson["ESS_0"],
        "ESS_1": Horvitz_Thompson["ESS_1"],
        "ESS_ATE": Horvitz_Thompson["ESS_ATE"],
    }

# 4 Generate Results

In [8]:
clip = 0.0

def process_pair(pair):

    ate_dict = get_ATE_wrapper(pair[0], pair[1], clip=clip)
    
    return {
        "cause_id": pair[0],
        "effect_id": pair[1],
        "ATE": ate_dict["ATE"],
        "STD": ate_dict["STD"],
        "size": ate_dict["size"],
        "ESS_0": ate_dict["ESS_0"],
        "ESS_1": ate_dict["ESS_1"],
        "ESS_ATE": ate_dict["ESS_ATE"],
        "ATE_ABLT": ate_dict["ATE_ABLT"],
        "STD_ABLT": ate_dict["STD_ABLT"],
        "ATE_STABILIZED": ate_dict["ATE_STABILIZED"],
        "STD_STABILIZED": ate_dict["STD_STABILIZED"],
        "cosine_similarity": cosimilarity(*pair),
        "correlation": correlation(*pair),
        "diff_of_conditionals": diff_of_conditionals(*pair),
        "jacard_index": jacard_index(*pair),
    }

In [9]:
from joblib import Parallel, delayed

all_results = Parallel(
    n_jobs=24,
    prefer="threads",
    return_as="generator_unordered",
)(
    delayed(process_pair)(pair)
    for pair in chosen_pairs_ids
)

all_results = tqdm(
    all_results,
    total=len(chosen_pairs_ids),
    smoothing=0.01,
)

raw_results = pd.DataFrame(all_results)

  0%|          | 0/9794 [00:00<?, ?it/s]

In [10]:
merged = pd.merge(
    left=oracle,
    right=raw_results,
    on=["cause_id", "effect_id"],
    how="inner",
)

merged.to_csv(data_path / 'sequels_evaluated.csv', index=False)